In [4]:
from datetime import datetime
import pandas as pd
from thefuzz import process, fuzz

fecha_diaria = datetime.now().strftime("%Y-%m-%d")

df_coolmod = pd.read_csv(f'port/portátiles_Coolmod_2026-04-13.csv')
df_pccomp = pd.read_csv(f'port/portátiles_Coolmod_2026-04-13.csv')

nombres_ref = df_pccomp['Nombre'].tolist()

df_coolmod['Match'] = df_coolmod['Nombre'].apply(
    lambda x: process.extractOne(x, nombres_ref, scorer=fuzz.token_set_ratio) if pd.notna(x) else None
)

df_coolmod['Nombre_13'] = df_coolmod['Match'].apply(lambda x: x[0] if x else None)
df_coolmod['Similitud'] = df_coolmod['Match'].apply(lambda x: x[1] if x else 0)

df_coolmod['Marca_Ok'] = df_coolmod.apply(
    lambda row: str(row['Nombre']).split()[0].upper() in str(row['Nombre_13']).upper() 
    if row['Nombre_13'] else False, axis=1
)

df_res = df_coolmod[(df_coolmod['Similitud'] >= 85) & (df_coolmod['Marca_Ok'])].copy()
df_pccomp_mini = df_pccomp[['Nombre', 'Precio']].rename(columns={'Nombre': 'Nombre_13', 'Precio': 'Precio_13'})

df_final = df_res.merge(df_pccomp_mini, on='Nombre_13')
df_final['Fecha'] = fecha_diaria
df_final['Diferencia'] = (df_final['Precio'] - df_final['Precio_13']).round(2)

df_final['MayorPrecio'] = df_final['Diferencia'].apply(
    lambda x: "Coolmod" if x > 0 else ("Día 13" if x < 0 else "Igual")
)
df_visualizacion = df_final[['Nombre', 'Precio', 'Precio_13', 'Similitud', 'Diferencia', 'Fecha', 'MayorPrecio']].copy()
df_visualizacion.columns = ['Producto', 'Precio Coolmod', 'Precio PcComp', 'Similitud (%)', 'Diferencia (€)', 'Fecha', 'MayorPrecio']
df_visualizacion.sort_values(by='Diferencia (€)', ascending=True).head(20)

,Producto,Precio Coolmod,Precio PcComp,Similitud (%),Diferencia (€),Fecha,MayorPrecio
219,Microsoft Surface Pro 11 Gen Copilot+ PC Snapd...,1391.95,1516.95,100,-125.0,2026-04-13,Día 13
169,Asus Vivobook 15 X1504VA-BQ575W Core 5 120U 16...,595.95,607.95,100,-12.0,2026-04-13,Día 13
0,Razer Cooling Pad RGB Inteligente 14 a 18,139.95,139.95,100,0.0,2026-04-13,Igual
1,Gigabyte AORUS Master 16 BYHC5ESE64SP U9-275HX...,3199.95,3199.95,100,0.0,2026-04-13,Igual
4,Medion Erazer Scout 17 E1 MD62758 C5-210H RTX ...,999.95,999.95,100,0.0,2026-04-13,Igual
5,Microsoft Surface Laptop 7 Copilot+ PC EP2-222...,1099.95,1099.95,100,0.0,2026-04-13,Igual
2,Gigabyte Gaming A16 CWHI3ES864SH i7-13620H RTX...,1349.95,1349.95,100,0.0,2026-04-13,Igual
3,Medion Erazer Scout 17 E1 MD62759 i5-13420H RT...,859.95,859.95,100,0.0,2026-04-13,Igual
8,MSI Vector 16 HX AI A2XWIG-CM (628XES) U9-275H...,2648.95,2648.95,100,0.0,2026-04-13,Igual
9,Gigabyte AERO X16 1VH93ESC94AH AMD Ryzen AI 7 ...,1399.95,1399.95,100,0.0,2026-04-13,Igual
